# Phase 11: Final Insights

This notebook summarizes the existing project outputs into business-facing insights and recommendations.

## 1. Data Loading

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_OUTPUTS = PROJECT_ROOT / 'data' / 'outputs'

merged_df = pd.read_csv(DATA_PROCESSED / 'merged_dataset.csv')
forecast_weekly = pd.read_csv(DATA_OUTPUTS / 'forecast_weekly.csv')
demand_classification = pd.read_csv(DATA_PROCESSED / 'demand_classification.csv')

merged_df['Week_End_Date'] = pd.to_datetime(merged_df['Week_End_Date'])
forecast_weekly['Week_End_Date'] = pd.to_datetime(forecast_weekly['Week_End_Date'])

print(f'Merged dataset: {merged_df.shape}')
print(f'Weekly forecast: {forecast_weekly.shape}')
print(f'Demand classification: {demand_classification.shape}')

Merged dataset: (2600, 21)
Weekly forecast: (300, 10)
Demand classification: (25, 8)


## 2. Category Analysis

In [2]:
category_analysis = (
    merged_df.groupby('Category', as_index=False)
    .agg(
        Total_Qty_Sold=('Qty_Sold', 'sum'),
        Avg_Weekly_Qty=('Qty_Sold', 'mean'),
        Total_Returns=('Returns', 'sum'),
        Avg_Discount=('Discount', 'mean'),
        Product_Count=('Product_ID', 'nunique')
    )
    .sort_values('Total_Qty_Sold', ascending=False)
)

category_analysis

,Category,Total_Qty_Sold,Avg_Weekly_Qty,Total_Returns,Avg_Discount,Product_Count
0,Apparel,63324,67.653846,2629,9.855769,9
2,Gear,49004,47.119231,1974,10.197115,10
1,Footwear,34454,55.214744,1391,9.863782,6


## 3. Top Products

In [3]:
top_products = (
    merged_df.groupby(['Product_ID', 'Product_Name', 'Category'], as_index=False)
    .agg(
        Total_Qty_Sold=('Qty_Sold', 'sum'),
        Avg_Weekly_Qty=('Qty_Sold', 'mean'),
        Total_Returns=('Returns', 'sum'),
        Avg_Inventory=('Inventory', 'mean')
    )
    .sort_values('Total_Qty_Sold', ascending=False)
    .head(10)
)

top_products

,Product_ID,Product_Name,Category,Total_Qty_Sold,Avg_Weekly_Qty,Total_Returns,Avg_Inventory
5,TE006,T-Shirt 6,Apparel,10090,97.019231,410,398.278846
18,TE019,T-Shirt 19,Apparel,10086,96.980769,445,400.125000
8,TE009,Pant 9,Apparel,9747,93.721154,424,397.519231
20,TE021,Hiking Boot 21,Footwear,9325,89.663462,413,398.461538
4,TE005,Accessorie 5,Gear,9300,89.423077,406,396.567308
14,TE015,Hiking Boot 15,Footwear,8046,77.365385,333,397.346154
10,TE011,T-Shirt 11,Apparel,8038,77.288462,354,397.355769
7,TE008,Accessorie 8,Gear,6882,66.173077,298,394.240385
3,TE004,Pant 4,Apparel,6735,64.759615,291,397.000000
17,TE018,Accessorie 18,Gear,6717,64.586538,302,397.692308


## 4. Demand Class Distribution

In [4]:
demand_class_distribution = (
    demand_classification['Demand_Class']
    .value_counts()
    .rename_axis('Demand_Class')
    .reset_index(name='Product_Count')
)
demand_class_distribution['Product_Share'] = (
    demand_class_distribution['Product_Count'] / demand_classification['Product_ID'].nunique()
)

demand_class_distribution

,Demand_Class,Product_Count,Product_Share
0,Smooth,25,1.0


## 5. Forecast Summary

In [5]:
forecast_summary = (
    forecast_weekly.groupby('Product_ID', as_index=False)
    .agg(
        Forecast_12_Week_Qty=('Forecast_Qty_Sold', 'sum'),
        Avg_Weekly_Forecast=('Forecast_Qty_Sold', 'mean'),
        Forecast_Start=('Week_End_Date', 'min'),
        Forecast_End=('Week_End_Date', 'max'),
        Demand_Class=('Demand_Class', 'first')
    )
    .sort_values('Forecast_12_Week_Qty', ascending=False)
)

forecast_summary.head(10)

,Product_ID,Forecast_12_Week_Qty,Avg_Weekly_Forecast,Forecast_Start,Forecast_End,Demand_Class
14,TE015,1552.293059,129.357755,2025-01-04,2025-03-22,Smooth
5,TE006,1552.152990,129.346082,2025-01-04,2025-03-22,Smooth
8,TE009,1549.980620,129.165052,2025-01-04,2025-03-22,Smooth
20,TE021,1547.876007,128.989667,2025-01-04,2025-03-22,Smooth
22,TE023,1352.067942,112.672328,2025-01-04,2025-03-22,Smooth
3,TE004,1343.328696,111.944058,2025-01-04,2025-03-22,Smooth
4,TE005,1212.732223,101.061019,2025-01-04,2025-03-22,Smooth
18,TE019,1141.352850,95.112738,2025-01-04,2025-03-22,Smooth
7,TE008,1131.685739,94.307145,2025-01-04,2025-03-22,Smooth
10,TE011,944.823744,78.735312,2025-01-04,2025-03-22,Smooth


In [6]:
overall_forecast = pd.DataFrame([{
    'Products': forecast_weekly['Product_ID'].nunique(),
    'Forecast_Weeks': forecast_weekly['Forecast_Step'].nunique(),
    'Forecast_Start': forecast_weekly['Week_End_Date'].min(),
    'Forecast_End': forecast_weekly['Week_End_Date'].max(),
    'Total_Forecast_Qty': forecast_weekly['Forecast_Qty_Sold'].sum(),
    'Avg_Weekly_Forecast_Qty': forecast_weekly['Forecast_Qty_Sold'].mean()
}])

overall_forecast

,Products,Forecast_Weeks,Forecast_Start,Forecast_End,Total_Forecast_Qty,Avg_Weekly_Forecast_Qty
0,25,12,2025-01-04,2025-03-22,22684.571625,75.615239


## 6. Business Insights

- Historical demand is concentrated in the highest-selling product categories and top products, so planning attention should prioritize those SKUs first.
- The demand classification output shows the product portfolio pattern used for modeling and reporting. This should be reviewed alongside forecast volume before inventory decisions.
- The 12-week forecast provides a forward-looking quantity estimate per product, useful for short-term replenishment and operational planning.
- Returns, discount levels, and inventory levels should be monitored for the top-selling categories because they have the largest impact on total business volume.

## 7. Recommendations

- Use the top product forecast table as the first input for near-term replenishment planning.
- Review high-volume categories weekly, especially when promotional activity or holiday weeks are present.
- Compare forecasted demand against available inventory and open orders before committing purchase or allocation decisions.
- Continue refreshing the pipeline as new weekly sales data becomes available so the lag, rolling, and forecast outputs stay current.

## Phase 11 Complete